# 2. Model Training & Hyperparameter Tuning
Models compared:
- **Logistic Regression** -- interpretable baseline, no tuning
- **XGBoost** -- tuned with Optuna (100 trials), `scale_pos_weight` tuned as a hyperparameter, optimized for PR-AUC
- **Random Forest** -- `class_weight='balanced_subsample'`, tuned with Optuna (100 trials)

Primary metric: **PR-AUC** (average precision) -- more sensitive to minority-class performance than ROC-AUC under ~9% positive-class imbalance.
Precision/Recall/F1 for each model are reported at that model's own F1-optimal threshold (not the default 0.5), since raw probabilities cluster well below 0.5 for all models here.

> Temporal split: train 2015-2022, test 2023-2025 (no data leakage).


In [1]:
import joblib
import numpy as np
import pandas as pd
import warnings
from datetime import datetime
import optuna
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    classification_report, recall_score, precision_score, f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import mlflow
import mlflow.sklearn

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('njit_dropout_prediction')

OUTPUT_DIR   = Path('outputs')
RANDOM_STATE = 42
CV_FOLDS     = 5
N_TRIALS     = 100

X_train = joblib.load(OUTPUT_DIR / 'X_train.pkl')
X_test  = joblib.load(OUTPUT_DIR / 'X_test.pkl')
y_train = joblib.load(OUTPUT_DIR / 'y_train.pkl')
y_test  = joblib.load(OUTPUT_DIR / 'y_test.pkl')

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Train positive rate: {y_train.mean():.4f} | Test: {y_test.mean():.4f}")


C:\Users\prati\Desktop\Project\ARR_NJIT_RD2\ARR_NJIT_RD2\ML_Model\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: (124463, 34) | Test: (65156, 34)
Train positive rate: 0.1072 | Test: 0.1065


## 2.1 Shared evaluation helper
Computes PR-AUC (threshold-independent) plus Precision/Recall/F1 at the model's own F1-optimal threshold.


In [2]:
def evaluate_model(name, model, X_test, y_test, cv_metric_value, cv_metric_std='-', stage='baseline'):
    y_prob = model.predict_proba(X_test)[:, 1]
    test_auc  = roc_auc_score(y_test, y_prob)
    pr_auc    = average_precision_score(y_test, y_prob)

    prec, rec, thresholds = precision_recall_curve(y_test, y_prob)
    f1_scores = 2 * prec * rec / (prec + rec + 1e-8)
    best_idx  = f1_scores[:-1].argmax()  # last point has no threshold
    opt_threshold = thresholds[best_idx]

    y_pred_opt = (y_prob >= opt_threshold).astype(int)
    precision  = precision_score(y_test, y_pred_opt, zero_division=0)
    recall     = recall_score(y_test, y_pred_opt, zero_division=0)
    f1         = f1_score(y_test, y_pred_opt, zero_division=0)

    print(f"{name}: CV={cv_metric_value:.4f} | Test AUC={test_auc:.4f} | PR-AUC={pr_auc:.4f} | "
          f"opt_thr={opt_threshold:.4f} | Precision={precision:.4f} | Recall={recall:.4f} | F1={f1:.4f}")

    # Log this model to MLflow: params, metrics, and the fitted model artifact
    with mlflow.start_run(run_name=name):
        raw_params = model.get_params()
        loggable_params = {k: v for k, v in raw_params.items() if isinstance(v, (str, int, float, bool)) or v is None}
        mlflow.log_params(loggable_params)
        mlflow.set_tags({"stage": stage, "algorithm": name, "run_date": datetime.now().strftime("%Y-%m-%d")})
        mlflow.log_metrics({
            "cv_pr_auc": float(cv_metric_value),
            "test_auc": float(test_auc),
            "pr_auc": float(pr_auc),
            "optimal_threshold": float(opt_threshold),
            "precision_at_risk": float(precision),
            "recall_at_risk": float(recall),
            "f1_at_risk": float(f1),
        })
        mlflow.sklearn.log_model(model, name="model", serialization_format="cloudpickle")

    return {
        "Model": name,
        "CV PR-AUC": round(cv_metric_value, 4),
        "CV Std": cv_metric_std if cv_metric_std == '-' else round(cv_metric_std, 4),
        "Test AUC": round(test_auc, 4),
        "PR-AUC": round(pr_auc, 4),
        "Optimal Threshold": round(float(opt_threshold), 4),
        "Precision (at-risk)": round(precision, 4),
        "Recall (at-risk)": round(recall, 4),
        "F1 (at-risk)": round(f1, 4),
    }


## 2.2 Baseline Models (default hyperparameters + class balancing)


In [3]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
pos_weight = (1 - y_train.mean()) / y_train.mean()

baseline_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            class_weight="balanced", max_iter=2000,
            solver="saga", random_state=RANDOM_STATE,
        )),
    ]),
}

results = []
fitted_models = {}

for name, model in baseline_models.items():
    cv_scores = cross_val_score(
        model, X_train, y_train, cv=cv, scoring="average_precision", n_jobs=-1
    )
    model.fit(X_train, y_train)
    results.append(evaluate_model(name, model, X_test, y_test, cv_scores.mean(), cv_scores.std(), stage="baseline"))
    fitted_models[name] = model

pd.DataFrame(results)


Logistic Regression: CV=0.1270 | Test AUC=0.5732 | PR-AUC=0.1273 | opt_thr=0.4897 | Precision=0.1267 | Recall=0.7789 | F1=0.2180


2026/07/04 22:04:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


,Model,CV PR-AUC,CV Std,Test AUC,PR-AUC,Optimal Threshold,Precision (at-risk),Recall (at-risk),F1 (at-risk)
0,Logistic Regression,0.127,0.0021,0.5732,0.1273,0.4897,0.1267,0.7789,0.218


## 2.3 Hyperparameter Tuning with Optuna
Logistic Regression is kept as an interpretable baseline with no tuning. XGBoost and Random Forest are each tuned with 100 Bayesian trials, optimizing **PR-AUC** (`average_precision`) via 3-fold CV. `scale_pos_weight` is tuned as a hyperparameter (not fixed) for XGBoost.


### 2.3.1 XGBoost


In [4]:
def objective_xgb(trial):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 100, 600),
        "max_depth":        trial.suggest_int("max_depth", 3, 10),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 5.0, 20.0),
        "eval_metric":      "aucpr",
        "random_state":     RANDOM_STATE,
        "verbosity":        0,
    }
    model = XGBClassifier(**params)
    scores = cross_val_score(
        model, X_train, y_train,
        cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
        scoring="average_precision", n_jobs=-1,
    )
    return scores.mean()

print("Tuning XGBoost with Optuna (100 trials, optimizing PR-AUC) ...")
study_xgb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, show_progress_bar=True)
print(f"Best CV PR-AUC: {study_xgb.best_value:.4f}")
print(f"Best params: {study_xgb.best_params}")


Tuning XGBoost with Optuna (100 trials, optimizing PR-AUC) ...


  0%|          | 0/100 [00:00<?, ?it/s]

Best trial: 0. Best value: 0.11674:   0%|          | 0/100 [00:15<?, ?it/s]

Best trial: 0. Best value: 0.11674:   1%|          | 1/100 [00:15<25:42, 15.58s/it]

Best trial: 1. Best value: 0.133884:   1%|          | 1/100 [00:31<25:42, 15.58s/it]

...

Best trial: 96. Best value: 0.136872:  99%|█████████▉| 99/100 [09:48<00:05,  5.52s/it]

Best trial: 96. Best value: 0.136872:  99%|█████████▉| 99/100 [09:53<00:05,  5.52s/it]

Best trial: 96. Best value: 0.136872: 100%|██████████| 100/100 [09:53<00:00,  5.45s/it]

Best trial: 96. Best value: 0.136872: 100%|██████████| 100/100 [09:53<00:00,  5.94s/it]

Best CV PR-AUC: 0.1369
Best params: {'n_estimators': 175, 'max_depth': 6, 'learning_rate': 0.010809646327535477, 'subsample': 0.94008823618446, 'colsample_bytree': 0.6876339909025658, 'min_child_weight': 7, 'gamma': 3.1645615907198157, 'scale_pos_weight': 13.026108756182351}


In [5]:
tuned_xgb = XGBClassifier(**study_xgb.best_params, eval_metric="aucpr", random_state=RANDOM_STATE, verbosity=0)
tuned_xgb.fit(X_train, y_train)

tuned_models = {}
tune_results = []

tuned_models["XGBoost (tuned)"] = tuned_xgb
tune_results.append(evaluate_model("XGBoost (tuned)", tuned_xgb, X_test, y_test, study_xgb.best_value, stage="tuned"))


XGBoost (tuned): CV=0.1369 | Test AUC=0.6097 | PR-AUC=0.1391 | opt_thr=0.5969 | Precision=0.1351 | Recall=0.7976 | F1=0.2310


2026/07/04 22:14:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


### 2.3.2 Random Forest


In [6]:
def objective_rf(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 600),
        "max_depth":         trial.suggest_int("max_depth", 4, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 30),
        "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 1, 30),
        "max_features":      trial.suggest_float("max_features", 0.3, 1.0),
        "class_weight":      "balanced_subsample",
        "random_state":      RANDOM_STATE,
        "n_jobs":            -1,
    }
    model = RandomForestClassifier(**params)
    scores = cross_val_score(
        model, X_train, y_train,
        cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
        scoring="average_precision", n_jobs=-1,
    )
    return scores.mean()

print("Tuning Random Forest with Optuna (100 trials, optimizing PR-AUC) ...")
study_rf = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_rf.optimize(objective_rf, n_trials=N_TRIALS, show_progress_bar=True)
print(f"Best CV PR-AUC: {study_rf.best_value:.4f}")
print(f"Best params: {study_rf.best_params}")


Tuning Random Forest with Optuna (100 trials, optimizing PR-AUC) ...


  0%|          | 0/100 [00:00<?, ?it/s]

Best trial: 0. Best value: 0.128476:   0%|          | 0/100 [00:46<?, ?it/s]

Best trial: 0. Best value: 0.128476:   1%|          | 1/100 [00:46<1:17:32, 46.99s/it]

Best trial: 1. Best value: 0.135573:   1%|          | 1/100 [01:06<1:17:32, 46.99s/it]

...

Best trial: 98. Best value: 0.13665:  99%|█████████▉| 99/100 [1:42:20<01:32, 92.67s/it]

Best trial: 98. Best value: 0.13665:  99%|█████████▉| 99/100 [1:43:51<01:32, 92.67s/it]

Best trial: 98. Best value: 0.13665: 100%|██████████| 100/100 [1:43:51<00:00, 92.22s/it]

Best trial: 98. Best value: 0.13665: 100%|██████████| 100/100 [1:43:51<00:00, 62.31s/it]

Best CV PR-AUC: 0.1366
Best params: {'n_estimators': 565, 'max_depth': 8, 'min_samples_split': 25, 'min_samples_leaf': 1, 'max_features': 0.950123145432255}


In [7]:
tuned_rf = RandomForestClassifier(
    **study_rf.best_params, class_weight="balanced_subsample",
    random_state=RANDOM_STATE, n_jobs=-1,
)
tuned_rf.fit(X_train, y_train)

tuned_models["Random Forest (tuned)"] = tuned_rf
tune_results.append(evaluate_model("Random Forest (tuned)", tuned_rf, X_test, y_test, study_rf.best_value, stage="tuned"))


Random Forest (tuned): CV=0.1366 | Test AUC=0.6096 | PR-AUC=0.1391 | opt_thr=0.4943 | Precision=0.1350 | Recall=0.7806 | F1=0.2302


2026/07/04 23:58:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


## 2.4 Model Comparison
Ranked by **PR-AUC** (threshold-independent, appropriate for this class imbalance).


In [8]:
all_results = pd.DataFrame(results + tune_results).sort_values("PR-AUC", ascending=False)
all_results.reset_index(drop=True)


,Model,CV PR-AUC,CV Std,Test AUC,PR-AUC,Optimal Threshold,Precision (at-risk),Recall (at-risk),F1 (at-risk)
0,XGBoost (tuned),0.1369,-,0.6097,0.1391,0.5969,0.1351,0.7976,0.2310
1,Random Forest (tuned),0.1366,-,0.6096,0.1391,0.4943,0.1350,0.7806,0.2302
2,Logistic Regression,0.1270,0.0021,0.5732,0.1273,0.4897,0.1267,0.7789,0.2180


In [9]:
# Pick overall best model by PR-AUC
best_row   = all_results.iloc[0]
best_name  = best_row['Model']
best_model = tuned_models.get(best_name, fitted_models.get(best_name))
best_threshold = best_row['Optimal Threshold']

print(f"Best model: {best_name}")
print(f"PR-AUC: {best_row['PR-AUC']}")
print(f"Test AUC: {best_row['Test AUC']}")
print(f"Optimal threshold: {best_threshold}")
print(f"Precision (at-risk): {best_row['Precision (at-risk)']}")
print(f"Recall (at-risk): {best_row['Recall (at-risk)']}")
print(f"F1 (at-risk): {best_row['F1 (at-risk)']}")

y_prob_best = best_model.predict_proba(X_test)[:, 1]
y_pred_best = (y_prob_best >= best_threshold).astype(int)
print("\nFull classification report (at optimal threshold):")
print(classification_report(y_test, y_pred_best,
                            target_names=['Returned (0)', 'At-Risk (1)']))


Best model: XGBoost (tuned)
PR-AUC: 0.1391
Test AUC: 0.6097
Optimal threshold: 0.5969
Precision (at-risk): 0.1351
Recall (at-risk): 0.7976
F1 (at-risk): 0.231

Full classification report (at optimal threshold):
              precision    recall  f1-score   support

Returned (0)       0.94      0.39      0.55     58219
 At-Risk (1)       0.14      0.80      0.23      6937

    accuracy                           0.43     65156
   macro avg       0.54      0.59      0.39     65156
weighted avg       0.86      0.43      0.52     65156



In [10]:
# Save best model + comparison table + its optimal threshold
all_results.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)
joblib.dump(best_model, OUTPUT_DIR / 'best_model.pkl')
joblib.dump(best_name,  OUTPUT_DIR / 'best_model_name.pkl')
joblib.dump(float(best_threshold), OUTPUT_DIR / 'optimal_threshold.pkl')

# Save all individual models
for name, m in {**fitted_models, **tuned_models}.items():
    safe = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    joblib.dump(m, OUTPUT_DIR / f'model_{safe}.pkl')

# Mark the winning run in MLflow so it is easy to find in the UI
best_run = mlflow.search_runs(
    filter_string=f"tags.algorithm = '{best_name}'",
    order_by=['start_time DESC'],
    max_results=1,
)
if not best_run.empty:
    with mlflow.start_run(run_id=best_run.iloc[0]['run_id']):
        mlflow.set_tag('selected_as_best', 'true')
        mlflow.log_artifact(str(OUTPUT_DIR / 'model_comparison.csv'))

print("Saved: best_model.pkl, best_model_name.pkl, optimal_threshold.pkl, model_comparison.csv, all individual models")
print(f'Best model logged to MLflow. Run: mlflow ui --backend-store-uri file:./mlruns  (then open http://127.0.0.1:5000)')
print("Next: run 03_evaluate.ipynb")


Saved: best_model.pkl, best_model_name.pkl, optimal_threshold.pkl, model_comparison.csv, all individual models
Best model logged to MLflow. Run: mlflow ui --backend-store-uri file:./mlruns  (then open http://127.0.0.1:5000)
Next: run 03_evaluate.ipynb
